# Lab 9 — ¿Tu entidad gana o pierde gente?

**Alumno:** Juan Diego Hernandez Zavala  
**Matrícula:** 2209978x  
**Entidad asignada:** Baja California  

> Resuelve las celdas marcadas con `TODO`. Cuando termines, descarga el notebook (`Archivo → Descargar → .ipynb`) y súbelo en la página de la actividad. No borres esta celda.


# Lab — Sesión 9: ¿Tu entidad gana o pierde gente?

**Simulación avanzada e implementación de modelos poblacionales y de catástrofe (ACD53)**

Sesión **EN CASA**. Cuatro partes. Se estima el saldo migratorio de tu entidad
por el **método residual**, se pasa todo a tasas, se verifica que suman, y —esto
es lo nuevo del curso— **se le pone una cota de error al resultado**.

La parte 3 es la que distingue este laboratorio: no pide un número, pide **un
número con su incertidumbre**. Es la primera vez que el curso lo exige, y va a
volver en las proyecciones de la unidad 4.

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ENTIDAD = "Baja California"   # <- tu entidad
FUENTE  = "INEGI: Encuesta Intercensal 2015 y Censo de Población y Vivienda 2020 (población total); Estadística de Nacimientos Registrados y Estadísticas de Defunciones Registradas, 2015-2019, por entidad de residencia habitual (tabulados interactivos Natalidad_01 y Mortalidad_01)"   # <- censos y registros: INEGI, años y tablas

## Datos

Cuatro cifras, y solo cuatro:

1. Población en el censo de **2015** (o 2010, si usas ese par).
2. Población en el censo de **2020**.
3. **Nacimientos** registrados en el periodo completo, sumando los años.
4. **Defunciones** registradas en el periodo completo.

Los datos de práctica son de una entidad **receptora** —le llega gente—, al
revés que la del material. No son datos reales de ninguna entidad.

In [60]:
ANIOS = 5      # duracion del periodo intercensal

# --- datos de PRACTICA (ilustrativos, no reales) ---------------------------
P0_PRACTICA = 1_860_000    # poblacion al inicio
P1_PRACTICA = 2_110_000    # poblacion al final
B_PRACTICA  =   195_000    # nacimientos en los 5 anios
D_PRACTICA  =    58_000    # defunciones en los 5 anios


def componentes(P0, P1, B, D, n=ANIOS):
    """Descompone el cambio de poblacion por la ecuacion compensadora.

    Devuelve un diccionario con el cambio total, el crecimiento natural, el
    saldo migratorio (por residual) y los persona-anios del periodo.
    """
    dP = P1 - P0
    CN = B - D
    return {'dP': dP, 'CN': CN, 'SM': dP - CN,
            'PA': n * (P0 + P1) / 2}

## Tus datos

In [61]:
# TODO 0: pega las cuatro cifras de tu entidad.
mi_P0 = 3_315_766    # Encuesta Intercensal 2015 (15 de marzo de 2015)
mi_P1 = 3_769_020    # Censo de Poblacion y Vivienda 2020 (15 de marzo de 2020)
mi_B  = sum([63_805, 62_608, 61_840, 60_174, 56_554])   # nacimientos registrados 2015-2019 (residencia de la madre)
mi_D  = sum([17_152, 18_253, 19_073, 20_501, 20_924])   # defunciones registradas 2015-2019 (residencia habitual)

if mi_P0 is None or mi_P1 is None or mi_B is None or mi_D is None:
    mi_P0, mi_P1 = P0_PRACTICA, P1_PRACTICA
    mi_B,  mi_D  = B_PRACTICA,  D_PRACTICA
    print("Usando los datos de PRACTICA. Sustituyelos por los de tu entidad.")
else:
    print(f"Usando datos propios: {ENTIDAD}")


Usando datos propios: Baja California


## Parte 1 — La descomposición  *(2.5 puntos)*

$$\Delta P = \underbrace{(B - D)}_{\text{natural}}
+ \underbrace{(I - E)}_{\text{migratorio}}
\qquad\Longrightarrow\qquad
\text{SM} = \Delta P - (B - D)$$

In [62]:
# TODO 1.1: calcula el cambio total, el crecimiento natural y el saldo
#   migratorio de tu entidad. Presentalos en una tabla con sus nombres.
comp = componentes(mi_P0, mi_P1, mi_B, mi_D)
df_comp = pd.DataFrame({
    'Componente': ['Población inicial (2015)', 'Población final (2020)', 'Nacimientos (B)',
                   'Defunciones (D)', 'Cambio total (ΔP)', 'Crecimiento natural (B - D)',
                   'Saldo migratorio neto (ΔP - (B - D))'],
    'Personas': [mi_P0, mi_P1, mi_B, mi_D, comp['dP'], comp['CN'], comp['SM']]
})
display(df_comp.style.format({'Personas': '{:,.0f}'}))

# TODO 1.2: di en una frase que tipo de entidad es la tuya. Hay cuatro casos
#   posibles segun los signos del crecimiento natural y del saldo migratorio:
#   escribe cual es el tuyo y que significa.
signo_cn = 'positivo' if comp['CN'] > 0 else 'negativo'
signo_sm = 'positivo' if comp['SM'] > 0 else 'negativo'
print(f"\nCrecimiento natural {signo_cn} y saldo migratorio {signo_sm}: {ENTIDAD} es una entidad "
      "receptora, crece por las dos vías. Nacen más personas de las que mueren y, además, llegan más "
      "personas de las que se van; la migración aporta incluso más que el crecimiento natural.")

# TODO 1.3: comprueba EN CODIGO que crecimiento natural mas saldo migratorio
#   reproduce el cambio total. Es trivial por construccion —y por eso mismo
#   sirve para atrapar un error de tecleo en los datos.
print(f"\nCN + SM = {comp['CN']:,} + {comp['SM']:,} = {comp['CN'] + comp['SM']:,}")
print(f"ΔP = P1 - P0 = {mi_P1:,} - {mi_P0:,} = {comp['dP']:,}")
print("¿Cuadra?:", comp['CN'] + comp['SM'] == mi_P1 - mi_P0)

,Componente,Personas
0,Población inicial (2015),"3,315,766"
1,Población final (2020),"3,769,020"
2,Nacimientos (B),"304,981"
3,Defunciones (D),"95,903"
4,Cambio total (ΔP),"453,254"
5,Crecimiento natural (B - D),"209,078"
6,Saldo migratorio neto (ΔP - (B - D)),"244,176"



Crecimiento natural positivo y saldo migratorio positivo: Baja California es una entidad receptora, crece por las dos vías. Nacen más personas de las que mueren y, además, llegan más personas de las que se van; la migración aporta incluso más que el crecimiento natural.

CN + SM = 209,078 + 244,176 = 453,254
ΔP = P1 - P0 = 3,769,020 - 3,315,766 = 453,254
¿Cuadra?: True


## Parte 2 — Las tasas  *(2.5 puntos)*

Todas sobre el mismo denominador: **persona-años** del periodo,
$\text{PA} = n \cdot (P_0 + P_1)/2$.

In [63]:
# TODO 2.1: calcula los persona-anios y las cinco tasas POR MIL Y POR ANIO:
#   natalidad, mortalidad, crecimiento natural, migracion neta y crecimiento
#   total. Presentalas en una tabla.
PA = comp['PA']
tbn = mi_B / PA * 1000
tbm = mi_D / PA * 1000
tcn = comp['CN'] / PA * 1000
tmn = comp['SM'] / PA * 1000
tct = comp['dP'] / PA * 1000
print(f"Persona-años del periodo: {PA:,.0f}")
df_tasas = pd.DataFrame({
    'Tasa': ['Natalidad', 'Mortalidad', 'Crecimiento natural', 'Migración neta', 'Crecimiento total'],
    'Por mil por año': [tbn, tbm, tcn, tmn, tct]
})
display(df_tasas.round(2))

# TODO 2.2: verifica EN CODIGO que la tasa de crecimiento natural mas la de
#   migracion neta dan la de crecimiento total. Imprime las tres y la
#   diferencia.
print(f"Crecimiento natural: {tcn:.4f} por mil")
print(f"Migración neta:      {tmn:.4f} por mil")
print(f"Crecimiento total:   {tct:.4f} por mil")
print(f"(natural + migración) - total = {tcn + tmn - tct:.2e} -> ¿cuadra?: {np.isclose(tcn + tmn, tct)}")

# TODO 2.3: calcula ademas la tasa de crecimiento por la ruta exponencial,
#   r = ln(P1/P0)/n, y comparala con la de la tabla. Si difieren, di por que:
#   las dos formulas coinciden solo cuando el crecimiento es pequenio.
r_exp = np.log(mi_P1 / mi_P0) / ANIOS
print(f"\nr exponencial = ln(P1/P0)/n = {r_exp*1000:.2f} por mil por año")
print(f"Tasa de crecimiento total (persona-años) = {tct:.2f} por mil por año")
print(f"Diferencia: {r_exp*1000 - tct:.3f} por mil ({(r_exp*1000/tct - 1)*100:.2f}%)")
print("Difieren muy poco, puesto que la de persona-años usa la media aritmética (P0+P1)/2 como denominador, "
      "y la exponencial equivale a usar la media logarítmica. Con un crecimiento de "
      f"{(mi_P1/mi_P0 - 1)*100:.1f}% en cinco años, las dos medias casi coinciden; la brecha crecería si el "
      "cambio en el periodo fuera grande.")

Persona-años del periodo: 17,711,965


,Tasa,Por mil por año
0,Natalidad,17.22
1,Mortalidad,5.41
2,Crecimiento natural,11.80
3,Migración neta,13.79
4,Crecimiento total,25.59


Crecimiento natural: 11.8043 por mil
Migración neta:      13.7859 por mil
Crecimiento total:   25.5903 por mil
(natural + migración) - total = -3.55e-15 -> ¿cuadra?: True

r exponencial = ln(P1/P0)/n = 25.63 por mil por año
Tasa de crecimiento total (persona-años) = 25.59 por mil por año
Diferencia: 0.035 por mil (0.14%)
Difieren muy poco, puesto que la de persona-años usa la media aritmética (P0+P1)/2 como denominador, y la exponencial equivale a usar la media logarítmica. Con un crecimiento de 13.7% en cinco años, las dos medias casi coinciden; la brecha crecería si el cambio en el periodo fuera grande.


## Parte 3 — El número, con su incertidumbre  *(2.5 puntos)*

El residual recoge la migración **y todos los errores** de los otros tres
números. Aquí se mide cuánto.

In [64]:
# TODO 3.1: repite el calculo del saldo migratorio suponiendo que el censo
#   final SUBESTIMO la poblacion en 1% y en 2%. Presenta los tres resultados
#   —0%, 1% y 2%— en una tabla con el cambio porcentual respecto al tuyo.
sm_base = comp['SM']
filas = []
for err in [0.00, 0.01, 0.02]:
    P1_verdadera = mi_P1 / (1 - err)          # si el censo contó solo el (1-err) de la población
    sm = componentes(mi_P0, P1_verdadera, mi_B, mi_D)['SM']
    filas.append({'Subconteo censo 2020': f"{err:.0%}", 'P1 corregida': P1_verdadera,
                  'Saldo migratorio neto': sm, 'Cambio vs. base (%)': (sm / sm_base - 1) * 100})
df_censo = pd.DataFrame(filas)
display(df_censo.round({'P1 corregida': 0, 'Saldo migratorio neto': 0, 'Cambio vs. base (%)': 2}))

# TODO 3.2: repite el calculo suponiendo un subregistro de nacimientos del 3%
#   (o sea, B verdadero = B registrado / 0.97). Compara ese efecto con el del
#   subconteo censal y di cual de los dos importa mas.
B_verdadero = mi_B / 0.97
sm_nac = componentes(mi_P0, mi_P1, B_verdadero, mi_D)['SM']
print(f"Con subregistro de nacimientos del 3%: B = {B_verdadero:,.0f}, saldo = {sm_nac:,.0f} "
      f"({(sm_nac / sm_base - 1) * 100:+.2f}% vs. base)")
print(f"Con subconteo censal del 1%: saldo = {df_censo.loc[1, 'Saldo migratorio neto']:,.0f} "
      f"({df_censo.loc[1, 'Cambio vs. base (%)']:+.2f}% vs. base)")
print(f"\nUn 3% de subregistro de nacimientos mueve el saldo en {abs(sm_nac - sm_base):,.0f} personas; "
      f"un 1% de subconteo del censo lo mueve en {abs(df_censo.loc[1, 'Saldo migratorio neto'] - sm_base):,.0f}. "
      "Importa más el error censal: se aplica sobre toda la población (3.8 millones), mientras que el de "
      "nacimientos solo se aplica sobre ~305 mil eventos. Además van en sentidos opuestos: el subconteo "
      "censal sube el saldo estimado y el subregistro de nacimientos lo baja.")

# TODO 3.3: calcula que porcentaje de tu poblacion final representa tu saldo
#   migratorio. Ese porcentaje es tu senial de alerta: si es del mismo orden
#   que el error de conteo plausible, tu estimacion es fragil. Dilo
#   explicitamente.
pct_sm = sm_base / mi_P1 * 100
print(f"\nEl saldo migratorio neto equivale al {pct_sm:.2f}% de la población de 2020.")
print(f"Un error de conteo plausible del censo es de 1-2%, varias veces menor que {pct_sm:.1f}%: "
      "el signo (entidad receptora) es robusto y no lo puede voltear ningún error razonable. La magnitud, en "
      f"cambio, sí es sensible: cada punto de subconteo censal mueve el saldo cerca de "
      f"{df_censo.loc[1, 'Cambio vs. base (%)']:.0f}%. Además, el censo 2015 es una encuesta (Intercensal) con error "
      "muestral propio, así que la estimación es firme en dirección pero imprecisa en tamaño.")

,Subconteo censo 2020,P1 corregida,Saldo migratorio neto,Cambio vs. base (%)
0,0%,3769020.0,244176.0,0.00
1,1%,3807091.0,282247.0,15.59
2,2%,3845939.0,321095.0,31.50


Con subregistro de nacimientos del 3%: B = 314,413, saldo = 234,744 (-3.86% vs. base)
Con subconteo censal del 1%: saldo = 282,247 (+15.59% vs. base)

Un 3% de subregistro de nacimientos mueve el saldo en 9,432 personas; un 1% de subconteo del censo lo mueve en 38,071. Importa más el error censal: se aplica sobre toda la población (3.8 millones), mientras que el de nacimientos solo se aplica sobre ~305 mil eventos. Además van en sentidos opuestos: el subconteo censal sube el saldo estimado y el subregistro de nacimientos lo baja.

El saldo migratorio neto equivale al 6.48% de la población de 2020.
Un error de conteo plausible del censo es de 1-2%, varias veces menor que 6.5%: el signo (entidad receptora) es robusto y no lo puede voltear ningún error razonable. La magnitud, en cambio, sí es sensible: cada punto de subconteo censal mueve el saldo cerca de 16%. Además, el censo 2015 es una encuesta (Intercensal) con error muestral propio, así que la estimación es firme en dirección pero i

## Parte 4 — El argumento, con sus cuatro piezas  *(2.5 puntos)*

In [65]:
# TODO 4.1 — EL RESULTADO
# Reporta tu saldo migratorio como RANGO, no como punto, usando los escenarios
# de la parte 3. Y usa el nombre correcto: saldo NETO, no emigrantes.
escenarios = [sm_base, sm_nac] + list(df_censo['Saldo migratorio neto'])
sm_min, sm_max = min(escenarios), max(escenarios)
print(f"Saldo migratorio neto de {ENTIDAD}, 2015-2020: entre {sm_min:,.0f} y {sm_max:,.0f} personas "
      f"(estimación central {sm_base:,.0f}).")
print(f"En tasa: entre {sm_min/PA*1000:.1f} y {sm_max/PA*1000:.1f} por mil por año. Es la diferencia "
      "entre los que llegaron y los que se fueron; no es un conteo de inmigrantes ni de emigrantes.")

Saldo migratorio neto de Baja California, 2015-2020: entre 234,744 y 321,095 personas (estimación central 244,176).
En tasa: entre 13.3 y 18.1 por mil por año. Es la diferencia entre los que llegaron y los que se fueron; no es un conteo de inmigrantes ni de emigrantes.


In [66]:
# TODO 4.2 — LA HISTORIA
# Explica, con tus dos tasas, por que tu entidad crece o decrece. Si crece a
# pesar de perder gente —o decrece a pesar de ganarla— dilo explicitamente: es
# el caso interesante y hay que nombrarlo.
print(f"{ENTIDAD} creció a {tct:.1f} por mil por año: {tcn:.1f} por mil de crecimiento natural más "
      f"{tmn:.1f} por mil de migración neta.")
print(f"No es el caso de signos opuestos, crece por las dos vías. Lo notable es que la migración "
      f"aporta el {tmn/tct*100:.0f}% del crecimiento, más que la diferencia entre nacimientos y defunciones.")
print("La TNR de la entidad es menor que 1, así que en el largo plazo las "
      "generaciones no se reemplazan por sí solas; el crecimiento natural positivo de hoy es sobre todo inercia de "
      "una estructura joven, y es la llegada de gente lo que sostiene el ritmo de crecimiento.")

Baja California creció a 25.6 por mil por año: 11.8 por mil de crecimiento natural más 13.8 por mil de migración neta.
No es el caso de signos opuestos, crece por las dos vías. Lo notable es que la migración aporta el 54% del crecimiento, más que la diferencia entre nacimientos y defunciones.
La TNR de la entidad es menor que 1, así que en el largo plazo las generaciones no se reemplazan por sí solas; el crecimiento natural positivo de hoy es sobre todo inercia de una estructura joven, y es la llegada de gente lo que sostiene el ritmo de crecimiento.


In [67]:
# TODO 4.3 — LO QUE EL RESIDUAL NO DISTINGUE
# Explica por que tu numero no dice cuanta gente se fue: es una diferencia entre
# entradas y salidas. Da un ejemplo numerico de dos situaciones distintas
# compatibles con TU saldo.
print(f"El residual solo da I - E = {sm_base:,.0f}. No dice cuántos entraron ni cuántos salieron: "
      "cualquier par de flujos con esa diferencia produce el mismo número.")
casos = pd.DataFrame({
    'Situación': ['A: poca movilidad', 'B: mucha rotación'],
    'Inmigrantes (I)': [sm_base + 60_000, sm_base + 400_000],
    'Emigrantes (E)': [60_000, 400_000],
})
casos['Saldo neto (I - E)'] = casos['Inmigrantes (I)'] - casos['Emigrantes (E)']
display(casos)
print("En A, la entidad casi solo recibe gente; en B, entra y sale muchísima (ciudad de paso hacia EE.UU., "
      "deportaciones, migración de retorno). Para Tijuana y Mexicali, el caso B es plausible, y el residual no puede "
      "separarlo del A.")

El residual solo da I - E = 244,176. No dice cuántos entraron ni cuántos salieron: cualquier par de flujos con esa diferencia produce el mismo número.


,Situación,Inmigrantes (I),Emigrantes (E),Saldo neto (I - E)
0,A: poca movilidad,304176,60000,244176
1,B: mucha rotación,644176,400000,244176


En A, la entidad casi solo recibe gente; en B, entra y sale muchísima (ciudad de paso hacia EE.UU., deportaciones, migración de retorno). Para Tijuana y Mexicali, el caso B es plausible, y el residual no puede separarlo del A.


In [68]:
# TODO 4.4 — QUE PEDIRIAS
# Di que dato adicional pedirias para darle mas credibilidad a tu estimacion, y
# por que ese en particular. La sesion propone cuatro caminos: revisalos y
# elige, no los enumeres todos.
print("Pediría la pregunta de LUGAR DE RESIDENCIA HACE CINCO AÑOS del Censo 2020 para Baja California "
      "(residentes en 2020 que en marzo de 2015 vivían en otra entidad o en otro país, y su contraparte: "
      "nacidos o residentes previos de BC que en 2020 viven en otra entidad).")
print("Es el dato que mide directamente los flujos de entrada y salida del mismo periodo 2015-2020, así que "
      "(1) permite separar I y E, que el residual no distingue, y (2) da una estimación independiente del saldo "
      "que no arrastra el error del conteo censal ni del registro de nacimientos: si coincide con el rango de la "
      "parte 3, la estimación gana credibilidad. Es especialmente valioso en una entidad fronteriza donde hay "
      "migración internacional además de la interna.")

Pediría la pregunta de LUGAR DE RESIDENCIA HACE CINCO AÑOS del Censo 2020 para Baja California (residentes en 2020 que en marzo de 2015 vivían en otra entidad o en otro país, y su contraparte: nacidos o residentes previos de BC que en 2020 viven en otra entidad).
Es el dato que mide directamente los flujos de entrada y salida del mismo periodo 2015-2020, así que (1) permite separar I y E, que el residual no distingue, y (2) da una estimación independiente del saldo que no arrastra el error del conteo censal ni del registro de nacimientos: si coincide con el rango de la parte 3, la estimación gana credibilidad. Es especialmente valioso en una entidad fronteriza donde hay migración internacional además de la interna.


## Cierre y entrega

**Checklist:**
- [ ] Nacimientos y defunciones son del **periodo completo**, no de un año.
- [ ] Las tasas usan **persona-años**, no la población de un año.
- [ ] La verificación de que las tasas suman está **impresa**.
- [ ] El saldo se reporta como **rango** y se llama **saldo neto**.
- [ ] Dije qué porcentaje de mi población es mi saldo, y qué implica.

**Cómo se entrega:** en Colab, `Archivo → Descargar → Descargar .ipynb`, y
arrastra ese archivo al sistema. **El archivo, no el enlace.** Ejecuta el
notebook completo antes de subirlo.

Tienes **dos intentos**. Y trae tus dudas por escrito al martes.